# Esta parte do código se refere à pipeline da camada BRONZE em BATCH para testes antes de subir ao AWS

In [43]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
# Instalando as dependências
# ~~~~~~~~~~~~~~~~~~~~~~~~~~

# basedosdados se refere a base que o Governo Brasileiro disponíbiliza para análises
# pyarrow para salvar em PARQUET

!pip install basedosdados pyarrow --quiet


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\carol\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [44]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
# Importações
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
import basedosdados as bd
import pandas as pd
import hashlib
import logging
import time
import os
from pathlib import Path

from datetime import datetime, timezone

In [45]:
# ~~~~~~~~~~~~~~~
# CONFIGURAÇÕES
# ~~~~~~~~~~~~~~~
PROJECT_ID = "tech-challenge-fase-2-502101"

DATASET = "br_inep_avaliacao_alfabetizacao"

TABELAS = [
    "uf",
    "meta_alfabetizacao_brasil",
    "meta_alfabetizacao_uf",
    "meta_alfabetizacao_municipio",
    "municipio",
    "alunos"
]

BUCKET = "fiap-alfabetizacao-ana-707472259268-us-east-1-an"

INGESTION_TS = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
INGESTION_DATE = datetime.now(timezone.utc).strftime("%Y-%m-%d")

CAMADA_BRONZE = "bronze"

In [46]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
# CONFIGURAÇÃO DOS LOGS
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%Y-%m-%dT%H:%M:%SZ",
)

log = logging.getLogger(__name__)

In [47]:
# ~~~~~~~~~~~~~~~
# LOG INICIAL
# ~~~~~~~~~~~~~~~

log.info("~" * 35)
log.info("INICIANDO ETL DA CAMADA BRONZE")
log.info(f"Projeto GCP : {PROJECT_ID}")
log.info(f"Dataset     : {DATASET}")
log.info(f"Bucket S3   : {BUCKET}")
log.info("~" * 35)

2026-07-12T15:20:04Z | INFO     | ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
2026-07-12T15:20:04Z | INFO     | INICIANDO ETL DA CAMADA BRONZE
2026-07-12T15:20:04Z | INFO     | Projeto GCP : tech-challenge-fase-2-502101
2026-07-12T15:20:04Z | INFO     | Dataset     : br_inep_avaliacao_alfabetizacao
2026-07-12T15:20:04Z | INFO     | Bucket S3   : fiap-alfabetizacao-ana-707472259268-us-east-1-an
2026-07-12T15:20:04Z | INFO     | ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


In [48]:
QUERIES = {

    "uf": """
    WITH
dicionario_serie AS (
    SELECT
        chave AS chave_serie,
        valor AS descricao_serie
    FROM `basedosdados.br_inep_avaliacao_alfabetizacao.dicionario`
    WHERE
        TRUE
        AND nome_coluna = 'serie'
        AND id_tabela = 'uf'
),
dicionario_rede AS (
    SELECT
        chave AS chave_rede,
        valor AS descricao_rede
    FROM `basedosdados.br_inep_avaliacao_alfabetizacao.dicionario`
    WHERE
        TRUE
        AND nome_coluna = 'rede'
        AND id_tabela = 'uf'
)
SELECT
    dados.ano as ano,
    dados.sigla_uf AS sigla_uf,
    diretorio_sigla_uf.nome AS sigla_uf_nome,
    descricao_serie AS serie,
    descricao_rede AS rede,
    dados.taxa_alfabetizacao as taxa_alfabetizacao,
    dados.media_portugues as media_portugues,
    dados.proporcao_aluno_nivel_0 as proporcao_aluno_nivel_0,
    dados.proporcao_aluno_nivel_1 as proporcao_aluno_nivel_1,
    dados.proporcao_aluno_nivel_2 as proporcao_aluno_nivel_2,
    dados.proporcao_aluno_nivel_3 as proporcao_aluno_nivel_3,
    dados.proporcao_aluno_nivel_4 as proporcao_aluno_nivel_4,
    dados.proporcao_aluno_nivel_5 as proporcao_aluno_nivel_5,
    dados.proporcao_aluno_nivel_6 as proporcao_aluno_nivel_6,
    dados.proporcao_aluno_nivel_7 as proporcao_aluno_nivel_7,
    dados.proporcao_aluno_nivel_8 as proporcao_aluno_nivel_8
FROM `basedosdados.br_inep_avaliacao_alfabetizacao.uf` AS dados
LEFT JOIN (SELECT DISTINCT sigla,nome  FROM `basedosdados.br_bd_diretorios_brasil.uf`) AS diretorio_sigla_uf
    ON dados.sigla_uf = diretorio_sigla_uf.sigla
LEFT JOIN `dicionario_serie`
    ON dados.serie = chave_serie
LEFT JOIN `dicionario_rede`
    ON dados.rede = chave_rede
    """,
    "meta_alfabetizacao_brasil": """ 
    SELECT
    dados.ano as ano,
    dados.rede as rede,
    dados.taxa_alfabetizacao as taxa_alfabetizacao,
    dados.meta_alfabetizacao_2024 as meta_alfabetizacao_2024,
    dados.meta_alfabetizacao_2025 as meta_alfabetizacao_2025,
    dados.meta_alfabetizacao_2026 as meta_alfabetizacao_2026,
    dados.meta_alfabetizacao_2027 as meta_alfabetizacao_2027,
    dados.meta_alfabetizacao_2028 as meta_alfabetizacao_2028,
    dados.meta_alfabetizacao_2029 as meta_alfabetizacao_2029,
    dados.meta_alfabetizacao_2030 as meta_alfabetizacao_2030,
    dados.percentual_participacao as percentual_participacao
FROM `basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_brasil` AS dados
    """,
     "meta_alfabetizacao_uf": """ 
    SELECT
    dados.ano as ano,
    dados.sigla_uf AS sigla_uf,
    diretorio_sigla_uf.nome AS sigla_uf_nome,
    dados.rede as rede,
    dados.taxa_alfabetizacao as taxa_alfabetizacao,
    dados.meta_alfabetizacao_2024 as meta_alfabetizacao_2024,
    dados.meta_alfabetizacao_2025 as meta_alfabetizacao_2025,
    dados.meta_alfabetizacao_2026 as meta_alfabetizacao_2026,
    dados.meta_alfabetizacao_2027 as meta_alfabetizacao_2027,
    dados.meta_alfabetizacao_2028 as meta_alfabetizacao_2028,
    dados.meta_alfabetizacao_2029 as meta_alfabetizacao_2029,
    dados.meta_alfabetizacao_2030 as meta_alfabetizacao_2030,
    dados.percentual_participacao as percentual_participacao
FROM `basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_uf` AS dados
LEFT JOIN (SELECT DISTINCT sigla,nome  FROM `basedosdados.br_bd_diretorios_brasil.uf`) AS diretorio_sigla_uf
    ON dados.sigla_uf = diretorio_sigla_uf.sigla
    """,

    "meta_alfabetizacao_municipio": """ 
      SELECT
      dados.ano as ano,
      dados.id_municipio AS id_municipio,
      diretorio_id_municipio.nome AS id_municipio_nome,
      dados.rede as rede,
      dados.taxa_alfabetizacao as taxa_alfabetizacao,
      dados.meta_alfabetizacao_2024 as meta_alfabetizacao_2024,
      dados.meta_alfabetizacao_2025 as meta_alfabetizacao_2025,
      dados.meta_alfabetizacao_2026 as meta_alfabetizacao_2026,
      dados.meta_alfabetizacao_2027 as meta_alfabetizacao_2027,
      dados.meta_alfabetizacao_2028 as meta_alfabetizacao_2028,
      dados.meta_alfabetizacao_2029 as meta_alfabetizacao_2029,
      dados.meta_alfabetizacao_2030 as meta_alfabetizacao_2030,
      dados.nivel_alfabetizacao as nivel_alfabetizacao,
      dados.percentual_participacao as percentual_participacao
  FROM `basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_municipio` AS dados
  LEFT JOIN (SELECT DISTINCT id_municipio,nome  FROM `basedosdados.br_bd_diretorios_brasil.municipio`) AS diretorio_id_municipio
      ON dados.id_municipio = diretorio_id_municipio.id_municipio
    """,

    "municipio": """ 
    WITH 
dicionario_serie AS (
    SELECT
        chave AS chave_serie,
        valor AS descricao_serie
    FROM `basedosdados.br_inep_avaliacao_alfabetizacao.dicionario`
    WHERE
        TRUE
        AND nome_coluna = 'serie'
        AND id_tabela = 'municipio'
),
dicionario_rede AS (
    SELECT
        chave AS chave_rede,
        valor AS descricao_rede
    FROM `basedosdados.br_inep_avaliacao_alfabetizacao.dicionario`
    WHERE
        TRUE
        AND nome_coluna = 'rede'
        AND id_tabela = 'municipio'
)
SELECT
    dados.ano as ano,
    dados.id_municipio AS id_municipio,
    diretorio_id_municipio.nome AS id_municipio_nome,
    descricao_serie AS serie,
    descricao_rede AS rede,
    dados.taxa_alfabetizacao as taxa_alfabetizacao,
    dados.media_portugues as media_portugues,
    dados.proporcao_aluno_nivel_0 as proporcao_aluno_nivel_0,
    dados.proporcao_aluno_nivel_1 as proporcao_aluno_nivel_1,
    dados.proporcao_aluno_nivel_2 as proporcao_aluno_nivel_2,
    dados.proporcao_aluno_nivel_3 as proporcao_aluno_nivel_3,
    dados.proporcao_aluno_nivel_4 as proporcao_aluno_nivel_4,
    dados.proporcao_aluno_nivel_5 as proporcao_aluno_nivel_5,
    dados.proporcao_aluno_nivel_6 as proporcao_aluno_nivel_6,
    dados.proporcao_aluno_nivel_7 as proporcao_aluno_nivel_7,
    dados.proporcao_aluno_nivel_8 as proporcao_aluno_nivel_8
FROM `basedosdados.br_inep_avaliacao_alfabetizacao.municipio` AS dados
LEFT JOIN (SELECT DISTINCT id_municipio,nome  FROM `basedosdados.br_bd_diretorios_brasil.municipio`) AS diretorio_id_municipio
    ON dados.id_municipio = diretorio_id_municipio.id_municipio
LEFT JOIN `dicionario_serie`
    ON dados.serie = chave_serie
LEFT JOIN `dicionario_rede`
    ON dados.rede = chave_rede
     """,

    "alunos": """ 
    WITH 
dicionario_serie AS (
    SELECT
        chave AS chave_serie,
        valor AS descricao_serie
    FROM `basedosdados.br_inep_avaliacao_alfabetizacao.dicionario`
    WHERE
        TRUE
        AND nome_coluna = 'serie'
        AND id_tabela = 'alunos'
),
dicionario_rede AS (
    SELECT
        chave AS chave_rede,
        valor AS descricao_rede
    FROM `basedosdados.br_inep_avaliacao_alfabetizacao.dicionario`
    WHERE
        TRUE
        AND nome_coluna = 'rede'
        AND id_tabela = 'alunos'
),
dicionario_presenca AS (
    SELECT
        chave AS chave_presenca,
        valor AS descricao_presenca
    FROM `basedosdados.br_inep_avaliacao_alfabetizacao.dicionario`
    WHERE
        TRUE
        AND nome_coluna = 'presenca'
        AND id_tabela = 'alunos'
),
dicionario_preenchimento_caderno AS (
    SELECT
        chave AS chave_preenchimento_caderno,
        valor AS descricao_preenchimento_caderno
    FROM `basedosdados.br_inep_avaliacao_alfabetizacao.dicionario`
    WHERE
        TRUE
        AND nome_coluna = 'preenchimento_caderno'
        AND id_tabela = 'alunos'
),
dicionario_alfabetizado AS (
    SELECT
        chave AS chave_alfabetizado,
        valor AS descricao_alfabetizado
    FROM `basedosdados.br_inep_avaliacao_alfabetizacao.dicionario`
    WHERE
        TRUE
        AND nome_coluna = 'alfabetizado'
        AND id_tabela = 'alunos'
)
SELECT
    dados.ano as ano,
    dados.id_municipio AS id_municipio,
    diretorio_id_municipio.nome AS id_municipio_nome,
    dados.id_escola as id_escola,
    dados.id_aluno as id_aluno,
    dados.caderno as caderno,
    descricao_serie AS serie,
    descricao_rede AS rede,
    descricao_presenca AS presenca,
    descricao_preenchimento_caderno AS preenchimento_caderno,
    descricao_alfabetizado AS alfabetizado,
    dados.proficiencia as proficiencia,
    dados.peso_aluno as peso_aluno
FROM `basedosdados.br_inep_avaliacao_alfabetizacao.alunos` AS dados
LEFT JOIN (SELECT DISTINCT id_municipio,nome  FROM `basedosdados.br_bd_diretorios_brasil.municipio`) AS diretorio_id_municipio
    ON dados.id_municipio = diretorio_id_municipio.id_municipio
LEFT JOIN `dicionario_serie`
    ON dados.serie = chave_serie
LEFT JOIN `dicionario_rede`
    ON dados.rede = chave_rede
LEFT JOIN `dicionario_presenca`
    ON dados.presenca = chave_presenca
LEFT JOIN `dicionario_preenchimento_caderno`
    ON dados.preenchimento_caderno = chave_preenchimento_caderno
LEFT JOIN `dicionario_alfabetizado`
    ON dados.alfabetizado = chave_alfabetizado
     """

}

In [ ]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# FUNÇÃO DE LEITURA DAS QUERIES
"""
    Executa uma consulta SQL na Base dos Dados utilizando o BigQuery.

    Args:
        query (str): Consulta SQL a ser executada.

    """
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
def ler_base_dados(query):

    df = bd.read_sql(
        query=query,
        billing_project_id=PROJECT_ID
    )

    return df

In [ ]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# CONSTRUINDO A CAMADA BRONZE
"""
    Adiciona metadados de ingestão ao DataFrame da camada Bronze.

    Args:
        df (pandas.DataFrame): Dados originais.
        dataset (str): Nome do dataset de origem.
        tabela (str): Nome da tabela de origem.
        
    """
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~

def construir_bronze(df, dataset, tabela):

    log.info("Adicionando metadados da camada Bronze")

    df = df.copy()

    df["_ingestion_timestamp"] = INGESTION_TS
    df["_ingestion_date"] = INGESTION_DATE
    df["_source_dataset"] = dataset
    df["_source_table"] = tabela

    df["_record_hash"] = (
        df.astype(str)
          .apply(lambda row: hashlib.md5("".join(row).encode()).hexdigest(), axis=1)
    )

    log.info(f"{len(df)} registros preparados para camada Bronze")

    return df

In [ ]:
# ~~~~~~~~~~~~~~~~~~~~~~~~
# REGRAS DE QUALIDADE
# ~~~~~~~~~~~~~~~~~~~~~~~~

CHECKS = {  
    "uf": [
        {"tipo": "min_count", "valor": 1, "critico": True},
        {"tipo": "not_null", "coluna": "ano", "critico": True},
        {"tipo": "not_null", "coluna": "sigla_uf", "critico": True},
        {"tipo": "not_null", "coluna": "sigla_uf_nome", "critico": True},
        {"tipo": "not_null", "coluna": "serie", "critico": True},
        {"tipo": "not_null", "coluna": "rede", "critico": True},
    ],

    "municipio": [
        {"tipo": "min_count", "valor": 1, "critico": True},
        {"tipo": "not_null", "coluna": "ano", "critico": True},
        {"tipo": "not_null", "coluna": "id_municipio", "critico": True},
        {"tipo": "not_null", "coluna": "id_municipio_nome", "critico": True},
        {"tipo": "not_null", "coluna": "serie", "critico": True},
        {"tipo": "not_null", "coluna": "rede", "critico": True},
    ],

    "meta_alfabetizacao_brasil": [
        {"tipo": "min_count", "valor": 1, "critico": True},
        {"tipo": "not_null", "coluna": "ano", "critico": True},
        {"tipo": "not_null", "coluna": "rede", "critico": True},
    ],

    "meta_alfabetizacao_uf": [
        {"tipo": "min_count", "valor": 1, "critico": True},
        {"tipo": "not_null", "coluna": "ano", "critico": True},
        {"tipo": "not_null", "coluna": "sigla_uf", "critico": True},
        {"tipo": "not_null", "coluna": "sigla_uf_nome", "critico": True},
        {"tipo": "not_null", "coluna": "rede", "critico": True},
    ],

    "meta_alfabetizacao_municipio": [
        {"tipo": "min_count", "valor": 1, "critico": True},
        {"tipo": "not_null", "coluna": "ano", "critico": True},
        {"tipo": "not_null", "coluna": "id_municipio", "critico": True},
        {"tipo": "not_null", "coluna": "id_municipio_nome", "critico": True},
        {"tipo": "not_null", "coluna": "rede", "critico": True},
    ],

    "alunos": [
        {"tipo": "min_count", "valor": 1, "critico": True},
        {"tipo": "not_null", "coluna": "ano", "critico": True},
        {"tipo": "not_null", "coluna": "id_aluno", "critico": True},
        {"tipo": "unique", "coluna": "id_aluno", "critico": False},
        {"tipo": "not_null", "coluna": "id_escola", "critico": True},
        {"tipo": "not_null", "coluna": "id_municipio", "critico": True},
        {"tipo": "not_null", "coluna": "serie", "critico": True},
        {"tipo": "not_null", "coluna": "rede", "critico": True},
        {"tipo": "not_null", "coluna": "presenca", "critico": True},
    ]
}

In [ ]:
# ~~~~~~~~~~~~~~
# DATA QUALITY
# ~~~~~~~~~~~~~~

def checar_qualidade(df, checks):
    """
    Executa as validações de qualidade da camada Bronze.

    Suporta os tipos: min_count, not_null e unique.
    Respeita o campo `critico`: se True (padrão), uma falha interrompe
    o pipeline (raise Exception); se False, a falha vira apenas um aviso
    no log (WARN) e a execução continua.

    Args:
        df (pandas.DataFrame): DataFrame da Bronze.
        checks (list): Lista de regras de validação.

    Raises:
        Exception: Caso alguma validação com critico=True falhe.
    """

    log.info("Iniciando verificações de qualidade")

    passou = 0
    falhou = 0

    for check in checks:

        tipo = check["tipo"]
        coluna = check.get("coluna")
        valor = check.get("valor")
        critico = check.get("critico", True)

        ok = False
        detalhe = ""

        if tipo == "min_count":

            quantidade = len(df)
            ok = quantidade >= valor
            detalhe = f"quantidade de registros={quantidade} | mínimo esperado={valor}"

        elif tipo == "not_null":

            nulos = df[coluna].isnull().sum()
            ok = nulos == 0
            detalhe = f"coluna '{coluna}' possui {nulos} valor(es) nulo(s)"

        elif tipo == "unique":

            duplicados = df.duplicated(subset=coluna).sum()
            ok = duplicados == 0
            detalhe = f"coluna(s) '{coluna}' possui(em) {duplicados} registro(s) duplicado(s)"

        else:

            log.warning(f"Tipo de check desconhecido, ignorado: '{tipo}'")
            continue

        status = "PASS" if ok else ("FAIL" if critico else "WARN")

        if ok:

            passou += 1
            log.info(f"[DQ:BRONZE] {status} | {tipo} | {detalhe}")

        else:

            falhou += 1

            if critico:
                log.error(f"[DQ:BRONZE] {status} | {tipo} | {detalhe}")
                raise Exception(f"Falha crítica de qualidade ({tipo}): {detalhe}")
            else:
                log.warning(f"[DQ:BRONZE] {status} | {tipo} | {detalhe}")

    log.info(f"Verificações de qualidade concluídas: {passou} passou(aram), {falhou} falhou(aram)")

In [ ]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# OBSERVABILIDADE: MÉTRICAS ESTRUTURADAS E ALERTAS
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Logging por si só não é observabilidade: para virar métrica consultável
# (ex.: no CloudWatch Logs Insights), o evento precisa ter uma estrutura
# previsível de campo=valor, em vez de texto livre solto na linha de log.

def log_metrica(evento, **campos):
    """
    Loga um evento estruturado (campo=valor), permitindo consulta via
    CloudWatch Logs Insights, ex.:
        fields @timestamp, tabela, volume, latencia_segundos
        | filter evento = "tabela_processada"

    Args:
        evento (str): Nome do evento (ex.: "tabela_processada", "pipeline_concluido").
        **campos: Pares chave=valor com os dados da métrica (volume, latência etc.).
    """
    campos_formatados = " | ".join(f"{chave}={valor}" for chave, valor in campos.items())
    log.info(f"[METRICA] evento={evento} | {campos_formatados}")


def emitir_alerta(mensagem, **contexto):
    """
    Emite um alerta de erro. Sempre loga em nível ERROR (visível em
    qualquer alarme de métrica de erro configurado sobre os logs do
    CloudWatch). Se houver um tópico SNS configurado (variável de
    ambiente SNS_TOPIC_ARN), também publica uma notificação -- para que
    a falha não dependa de alguém abrir o log manualmente (ex.: uma
    carga que falha num sábado).

    Args:
        mensagem (str): Descrição do alerta.
        **contexto: Dados adicionais para diagnóstico (tabela, etapa, erro etc.).
    """
    contexto_formatado = " | ".join(f"{k}={v}" for k, v in contexto.items())
    log.error(f"[ALERTA] {mensagem} | {contexto_formatado}")

    topico_sns = os.environ.get("SNS_TOPIC_ARN")

    if not topico_sns:
        log.warning("[ALERTA] SNS_TOPIC_ARN não configurado - alerta ficou registrado apenas no log")
        return

    try:
        import boto3
        sns = boto3.client("sns")
        sns.publish(
            TopicArn=topico_sns,
            Subject="[Tech Challenge] Falha no pipeline",
            Message=f"{mensagem}\n\n{contexto_formatado}"
        )
        log.info("[ALERTA] Notificação SNS publicada com sucesso")
    except Exception as e:
        log.warning(f"[ALERTA] Falha ao publicar no SNS (alerta permanece apenas no log): {e}")

In [ ]:
# ~~~~~~~~~~~~~~~~~~~~
# SALVAR CAMADA BRONZE
# ~~~~~~~~~~~~~~~~~~~~

def salvar_bronze(df, tabela):

    # Particiona por ingestion_date (padrão Hive: chave=valor), para que
    # cada execução crie uma pasta nova em vez de sobrescrever a anterior.
    # Isso preserva o histórico completo de cargas, como o README promete,
    # e já deixa o layout pronto para um Glue Crawler detectar partições
    # automaticamente (particionamento físico em Parquet).
    pasta = Path("bronze") / tabela / f"ingestion_date={INGESTION_DATE}"
    pasta.mkdir(parents=True, exist_ok=True)

    arquivo = pasta / f"{tabela}.parquet"

    log.info(f"Salvando arquivo: {arquivo}")

    df.to_parquet(
        arquivo,
        index=False,
        engine="pyarrow"
    )

    log.info("Arquivo Parquet criado com sucesso.")

    return arquivo

In [54]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# FUNÇÃO PARA CAMADA BRONZE PARA REUTILIZAR EM VÁRIAS TABELAS
"""
    Executa o pipeline completo da camada Bronze para todas as tabelas
    configuradas no dicionário QUERIES.

    Etapas:
        1. Leitura da Base dos Dados.
        2. Construção da camada Bronze.
        3. Validação de qualidade dos dados.
        4. Geração do arquivo Parquet.

    Cada tabela é processada de forma isolada: se uma tabela falhar, um
    alerta é emitido e as demais continuam sendo processadas (a falha de
    uma tabela não derruba o pipeline inteiro). Latência e volume de cada
    tabela são registrados como métricas estruturadas e consultáveis.
"""
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
def executar_bronze():

    inicio_pipeline = time.perf_counter()

    tabelas_ok = 0
    tabelas_falha = 0

    for tabela, query in QUERIES.items():

        log.info("~" * 35)
        log.info(f"Iniciando execução da camada Bronze para '{tabela}'")
        log.info("~" * 35)

        inicio_tabela = time.perf_counter()

        try:

            # leitura da base dos dados
            df = ler_base_dados(query)

            # construindo a bronze com os metadados
            df_bronze = construir_bronze(df, DATASET, tabela)

            # exibe as primeiras linhas, somente usado no colab
            print(f"\nPrévia da tabela: {tabela}")
            display(df_bronze.head())

            # checks de integridades e tipos
            checks = CHECKS.get(tabela, [])

            # data quality
            if checks:
                checar_qualidade(df_bronze, checks)

            # salvando
            salvar_bronze(df_bronze, tabela)

            latencia_segundos = round(time.perf_counter() - inicio_tabela, 2)

            log_metrica(
                "tabela_processada",
                camada="bronze",
                tabela=tabela,
                volume=len(df_bronze),
                latencia_segundos=latencia_segundos,
                status="sucesso"
            )

            tabelas_ok += 1

        except Exception as e:

            latencia_segundos = round(time.perf_counter() - inicio_tabela, 2)

            log_metrica(
                "tabela_processada",
                camada="bronze",
                tabela=tabela,
                volume=0,
                latencia_segundos=latencia_segundos,
                status="falha"
            )

            emitir_alerta(
                f"Falha na ingestão da tabela '{tabela}' na camada Bronze",
                camada="bronze",
                tabela=tabela,
                erro=str(e)
            )

            tabelas_falha += 1

            # isola a falha: segue para a próxima tabela em vez de
            # derrubar o restante do pipeline
            continue

    latencia_total_segundos = round(time.perf_counter() - inicio_pipeline, 2)

    log_metrica(
        "pipeline_concluido",
        camada="bronze",
        tabelas_ok=tabelas_ok,
        tabelas_falha=tabelas_falha,
        latencia_total_segundos=latencia_total_segundos
    )

    if tabelas_falha > 0:
        emitir_alerta(
            f"Pipeline Bronze concluído com {tabelas_falha} falha(s) de {tabelas_ok + tabelas_falha} tabela(s)",
            camada="bronze",
            tabelas_falha=tabelas_falha,
            tabelas_ok=tabelas_ok
        )

    log.info("Camada Bronze concluída!")

2026-07-12T15:20:04Z | INFO     | Camada Bronze concluída com sucesso!


In [55]:
executar_bronze()

2026-07-12T15:20:04Z | INFO     | ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
2026-07-12T15:20:04Z | INFO     | Iniciando execução da camada Bronze para 'uf'
2026-07-12T15:20:04Z | INFO     | ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


Downloading: 100%|██████████|

2026-07-12T15:20:06Z | INFO     | Adicionando metadados da camada Bronze
2026-07-12T15:20:06Z | INFO     | 145 registros preparados para camada Bronze




Prévia da tabela: uf


,ano,sigla_uf,sigla_uf_nome,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,...,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8,_ingestion_timestamp,_ingestion_date,_source_dataset,_source_table,_record_hash
0,2023,AM,Amazonas,2° ano do Ensino Fundamental,Municipal,49.20,733.6637,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,20260712_182003,2026-07-12,br_inep_avaliacao_alfabetizacao,uf,006c3dd232dd2d714c8b91fcdb213597
1,2023,PB,Paraíba,2° ano do Ensino Fundamental,Estadual,55.23,744.8152,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,20260712_182003,2026-07-12,br_inep_avaliacao_alfabetizacao,uf,d1723c8211486d0f49890b14de2e8935
2,2023,PR,Paraná,2° ano do Ensino Fundamental,Pública (Estadual e Municipal),73.12,757.2146,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,20260712_182003,2026-07-12,br_inep_avaliacao_alfabetizacao,uf,aeb6b5b5f9068b9f90f3b0c109d92a12
3,2023,AP,Amapá,2° ano do Ensino Fundamental,Municipal,41.87,732.7858,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,20260712_182003,2026-07-12,br_inep_avaliacao_alfabetizacao,uf,72cbf9d556e8b41386fcd1f7a42d9611
4,2023,PE,Pernambuco,2° ano do Ensino Fundamental,Pública (Estadual e Municipal),58.95,747.4522,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,20260712_182003,2026-07-12,br_inep_avaliacao_alfabetizacao,uf,f331cca0113c0ded834b0743003fe226


...

Downloading: 100%|██████████|


2026-07-12T15:33:26Z | INFO     | Total time taken 784.43 s.
Finished at 2026-07-12 15:33:26.
2026-07-12T15:33:26Z | INFO     | Adicionando metadados da camada Bronze
2026-07-12T15:34:03Z | INFO     | 3867999 registros preparados para camada Bronze



Prévia da tabela: alunos


,ano,id_municipio,id_municipio_nome,id_escola,id_aluno,caderno,serie,rede,presenca,preenchimento_caderno,alfabetizado,proficiencia,peso_aluno,_ingestion_timestamp,_ingestion_date,_source_dataset,_source_table,_record_hash
0,2024,3204906,São Mateus,60022379,32028878,43,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN,20260712_182003,2026-07-12,br_inep_avaliacao_alfabetizacao,alunos,baea6c3c62cebec2aa7d9764f7ad5892
1,2024,3205309,Vitória,60022713,32007513,43,2° ano do Ensino Fundamental,Municipal,Presente,Prova preenchida,Sim,786.93,1.09,20260712_182003,2026-07-12,br_inep_avaliacao_alfabetizacao,alunos,0f88a17e6b5843f31dd49483b5e4d235
2,2023,1302603,Manaus,60000703,13024170,1,2° ano do Ensino Fundamental,Estadual,Ausente,Prova não preenchida,Não,NaN,NaN,20260712_182003,2026-07-12,br_inep_avaliacao_alfabetizacao,alunos,90f3659273bca119cf2b6135e2987fcf
3,2023,1501808,Breves,60001609,15031556,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN,20260712_182003,2026-07-12,br_inep_avaliacao_alfabetizacao,alunos,2e1384aa64d67701a4251512bd0146db
4,2023,1502806,Curralinho,60001620,15041830,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN,20260712_182003,2026-07-12,br_inep_avaliacao_alfabetizacao,alunos,7c3ac3c295ac6c389f65784295de81cd


2026-07-12T15:34:03Z | INFO     | Iniciando verificações de qualidade
2026-07-12T15:34:03Z | INFO     | OK - Quantidade de registros: 3867999
2026-07-12T15:34:03Z | INFO     | OK - Coluna 'ano' sem valores nulos
2026-07-12T15:34:04Z | INFO     | OK - Coluna 'id_aluno' sem valores nulos
2026-07-12T15:34:04Z | INFO     | OK - Coluna 'id_escola' sem valores nulos
2026-07-12T15:34:04Z | INFO     | OK - Coluna 'id_municipio' sem valores nulos
2026-07-12T15:34:04Z | INFO     | OK - Coluna 'serie' sem valores nulos
2026-07-12T15:34:04Z | INFO     | OK - Coluna 'rede' sem valores nulos
2026-07-12T15:34:04Z | INFO     | OK - Coluna 'presenca' sem valores nulos
2026-07-12T15:34:04Z | INFO     | Todas as verificações passaram com sucesso.
2026-07-12T15:34:04Z | INFO     | Salvando arquivo: bronze\alunos.parquet
2026-07-12T15:34:12Z | INFO     | Arquivo Parquet criado com sucesso.
